In [1]:
pip install pyspark

In [1]:
from google.colab import auth
auth.authenticate_user()
print("✅ Autenticado con Google Cloud!")

✅ Autenticado con Google Cloud!


In [2]:
from pyspark.sql import SparkSession

print("⚙️ Encendiendo el motor de Apache Spark...")

spark = SparkSession.builder \
    .appName("CryptoPipeline") \
    .config("spark.jars.packages", "com.google.cloud.bigdataoss:gcs-connector:hadoop3-2.2.5") \
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem") \
    .config("spark.hadoop.fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS") \
    .getOrCreate()

print("✅ Motor listo. ¡Spark usará tu cuenta oficial de Google!")

⚙️ Encendiendo el motor de Apache Spark...
✅ Motor listo. ¡Spark usará tu cuenta oficial de Google!


In [3]:
# Reemplaza con el nombre exacto de tu bucket
bucket_name = "brayan-data-lake-olist"
ruta_datos = f"gs://{bucket_name}/raw/crypto/"

print(f"📡 Leyendo miles de archivos desde: {ruta_datos}...")

# Spark lee TODOS los archivos Parquet de esa carpeta en un solo golpe
df_spark = spark.read.parquet(ruta_datos)

# Mostramos las primeras 20 filas
df_spark.show()

📡 Leyendo miles de archivos desde: gs://brayan-data-lake-olist/raw/crypto/...
+--------+-------+--------------------+--------------------+-------------------+
|  moneda|    usd|      usd_market_cap|         usd_24h_vol|   fecha_extraccion|
+--------+-------+--------------------+--------------------+-------------------+
| bitcoin|71167.0|1.426495813930944...|3.786616801476691E10|2026-06-01 10:46:51|
|ethereum|1971.34|2.379331905351197...|1.547569342638853E10|2026-06-01 10:46:51|
|  solana|  79.76|4.611724380089938E10|2.3552087802197695E9|2026-06-01 10:46:51|
| bitcoin|71040.0|1.422602237134134...|4.029050735641127E10|2026-06-01 11:02:05|
|ethereum|1966.81|2.372417747182552E11|1.623866987759428...|2026-06-01 11:02:05|
|  solana|  79.56|4.604651233846614E10|2.3609945630217333E9|2026-06-01 11:02:05|
| bitcoin|60922.0|1.221126222047213...|7.429650366482816E10|2026-06-06 00:00:26|
|ethereum|1580.56|1.906748841672772E11|3.942148200185436E10|2026-06-06 00:00:26|
|  solana|  63.48|3.67182681024

In [4]:
from pyspark.sql import functions as sf

df_resumen = df_spark.groupBy("moneda").agg(sf.round(sf.avg("usd"), 2).alias("precio"), sf.max("usd_24h_vol").alias("volumen"))

df_resumen.show()


+--------+--------+--------------------+
|  moneda|  precio|             volumen|
+--------+--------+--------------------+
|  solana|   64.96| 6.996828650977853E9|
| bitcoin|62271.11|7.429650366482816E10|
|ethereum| 1625.31|3.942148200185436E10|
+--------+--------+--------------------+



In [6]:
# 1. Convertimos la tabla de Spark (que vive en múltiples nodos) a una tabla de Pandas (memoria local)
df_pandas = df_resumen.toPandas()

print("🚀 Iniciando viaje desde Colab hacia BigQuery...")

# 2. Usamos la función nativa de Pandas para inyectar los datos a BigQuery
df_pandas.to_gbq(
    destination_table='crypto_analytics.resumen_monedas',
    project_id='project-59fdbdb5-71ea-4b99-835',
    if_exists='replace' # Si la tabla ya existe, la sobrescribe con los datos frescos
)

print("✅ ¡Aterrizaje perfecto! Los datos ya están en la capa Gold de GCP.")

🚀 Iniciando viaje desde Colab hacia BigQuery...


/tmp/ipykernel_32574/154444375.py:7: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  df_pandas.to_gbq(
100%|██████████| 1/1 [00:00<00:00, 7557.30it/s]

✅ ¡Aterrizaje perfecto! Los datos ya están en la capa Gold de GCP.
